In [ ]:
!pip3 install ucimlrepo

: 

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
wine_quality = fetch_ucirepo(id=186)

# data (as pandas dataframes)
X = wine_quality.data.features
y = wine_quality.data.targets

# metadata
print(wine_quality.metadata)

# variable information
print(wine_quality.variables)


## Helper Functions ##

In [ ]:
def _is_nan(x):
    # works for python floats and numpy float types
    return x != x

In [ ]:
def _safe_float(x):
    # convert to float if possible; keep as-is otherwise
    try:
        return float(x)
    except:
        return x

In [ ]:
def _col_as_list(data, j):
    n = data.shape[0]
    col = [0.0] * n
    for i in range(n):
        col[i] = data[i, j]
    return col

In [ ]:
def _sum_1d(arr):
    s = 0.0
    for v in arr:
        s += v
    return s

In [ ]:
def _mean_1d(arr):
    n = len(arr)
    if n == 0:
        raise ValueError("Cannot compute mean of empty array")
    return _sum_1d(arr) / n

## Problem 1

a) What is the data used for (explain this in a well-written paragraph)? [5 points]

The data selected aims to model wine quality. This is done base on the results of physicochemical tests

b) Who (or what organization) uploaded the data? [1 point]

Science Direct posted it, but the creator is Paulo Cortez, Antonio Cerdeira, Fernando Almedia, Telmo Matos, and Jose Reis from the department of Information Systems / R&D Centre Algoritmi, University of Minho and Viticulture Commission of the Vinho Verde Region. 

c) How many attributes and how many entries are included in the data?
a. How many numerical attributes? [1 point]
b. How many categorical attributes? [1 point]
c. For the categorical attributes, would you suggest integer encoding (label encoding) or
one-hot encoding? And why? Answer this for each categorical attribute. [3 points]
 There are 13 attributes with 4898 instances. There are 12 numerical attributes and 1 categorical attribute. We would probably do one hot encoding for our attribute becuase it is one value or the other - red or white win. 

d) Missing values:

a. Are there missing values in this dataset? [1 point]

None

b. If so, for each attribute, what is the proportion of the data that is missing? [4 points]

None

c. What is the proportion of the missing data overall? (You can use the plots to summarize
this data and explain each answer). [1 point]

None

e) What is the most interesting thing about this dataset to you? Explain this in a well-written
paragraph. [6 points]

The variety of wine that is able to be explained by the attributes is quite interesting. Being able to classify each wine by "color" is quite a vague classification but looking at the chemical properties of each wine that contains different types of grapes, while also being able to profile these wines together despite their similarities is also interesting. 

f) Out of all the attributes in this dataset, what do you think are the most descriptive attributes?
(before doing any data analysis)? [5 points]

Red or white, they are the most descriptive and will most likely be key factors in our data set. 



## Problem 2 
a) Write a function that computes the multi-dimensional mean of the numerical attributes. You should return the answer as a 2D NumPy array. [5 points]


In [ ]:
# returns 2D numpy array: (1 x d)
def multidimensional_mean(data_2d):
    n, d = data_2d.shape
    means = np.zeros((1, d), dtype=float)

    for j in range(d):
        s = 0.0
        for i in range(n):
            s += float(data_2d[i, j])
        means[0, j] = s / n

    return means

b) A function to calculate sample variance of a given attribute. [5 points]

In [ ]:
# s^2 = (1/(n-1)) * sum (xi - xbar)^2
def sample_variance(attribute_1d):
    n = len(attribute_1d)
    if n < 2:
        raise ValueError("Sample variance requires n >= 2")

    # mean manually
    s = 0.0
    for i in range(n):
        s += float(attribute_1d[i])
    mean = s / n

    # squared deviations
    ss = 0.0
    for i in range(n):
        diff = float(attribute_1d[i]) - mean
        ss += diff * diff

    return ss / (n - 1)

c) A function to calculate sample covariance between two given attributes. [5 points]

In [ ]:
# cov(X,Y) = (1/(n-1))*sum (xi-xbar)(yi-ybar)
def sample_covariance(attr1_1d, attr2_1d):
    n = len(attr1_1d)
    if n != len(attr2_1d):
        raise ValueError("Attributes must have same length")
    if n < 2:
        raise ValueError("Sample covariance requires n >= 2")

    # means manually
    s1 = 0.0
    s2 = 0.0
    for i in range(n):
        s1 += float(attr1_1d[i])
        s2 += float(attr2_1d[i])
    m1 = s1 / n
    m2 = s2 / n

    # covariance sum
    sxy = 0.0
    for i in range(n):
        sxy += (float(attr1_1d[i]) - m1) * (float(attr2_1d[i]) - m2)

    return sxy / (n - 1)

d) A function that calculates the covariance matrix. (You can use the functions that you have
created in b and c). [5 points]


In [ ]:
# returns d x d numpy array
def covariance_matrix(data_2d):
    n, d = data_2d.shape
    cov = np.zeros((d, d), dtype=float)

    # pre-extract columns to python lists
    cols = []
    for j in range(d):
        cols.append(_col_as_list(data_2d, j))

    for i in range(d):
        for j in range(d):
            cov[i, j] = sample_covariance(cols[i], cols[j])

    return cov

e) A function to calculate the correlation coefficient between two attributes. [5 points]


In [ ]:
# corr = cov / (std1 * std2), std = sqrt(variance)
def correlation_coefficient(attr1_1d, attr2_1d):
    cov = sample_covariance(attr1_1d, attr2_1d)
    var1 = sample_variance(attr1_1d)
    var2 = sample_variance(attr2_1d)

    if var1 == 0.0 or var2 == 0.0:
        raise ValueError("Correlation undefined when variance is zero")

    std1 = var1 ** 0.5
    std2 = var2 ** 0.5
    return cov / (std1 * std2)

f) A function that normalizes the attributes in a 2D NumPy array using z-score normalization. [5
points]


In [ ]:
# z = (x - mean) / std  (use sample std with ddof=1)
def z_score_normalization(data_2d):
    n, d = data_2d.shape
    out = np.zeros((n, d), dtype=float)

    # compute per-column mean and sample std manually
    means = [0.0] * d
    stds  = [0.0] * d

    for j in range(d):
        # mean
        s = 0.0
        for i in range(n):
            s += float(data_2d[i, j])
        means[j] = s / n

        # sample variance
        ss = 0.0
        for i in range(n):
            diff = float(data_2d[i, j]) - means[j]
            ss += diff * diff
        if n < 2:
            raise ValueError("Need n >= 2 for sample std")
        var = ss / (n - 1)
        stds[j] = var ** 0.5

        if stds[j] == 0.0:
            # if constant column, z-score would divide by zero
            # you can set to 0, or raise error. I set to 0.
            stds[j] = 0.0

    # normalize
    for i in range(n):
        for j in range(d):
            if stds[j] == 0.0:
                out[i, j] = 0.0
            else:
                out[i, j] = (float(data_2d[i, j]) - means[j]) / stds[j]

    return out

g) A function that normalizes the attributes in a 2D NumPy array using range normalization. [5
points]


In [ ]:
# r = (x - min) / (max - min)
def range_normalization(data_2d):
    n, d = data_2d.shape
    out = np.zeros((n, d), dtype=float)

    mins = [0.0] * d
    maxs = [0.0] * d

    for j in range(d):
        # initialize min/max from first element
        mn = float(data_2d[0, j])
        mx = float(data_2d[0, j])

        for i in range(1, n):
            v = float(data_2d[i, j])
            if v < mn:
                mn = v
            if v > mx:
                mx = v

        mins[j] = mn
        maxs[j] = mx

    for i in range(n):
        for j in range(d):
            denom = (maxs[j] - mins[j])
            if denom == 0.0:
                out[i, j] = 0.0
            else:
                out[i, j] = (float(data_2d[i, j]) - mins[j]) / denom

    return out

h) A function that will compute the covariance matrix of a dataset. [5 points]

In [ ]:
# (This is essentially (d) again;
def compute_covariance_matrix(data_2d):
    return covariance_matrix(data_2d)

i) A function that will label-encode a 2D categorical data array that is passed as an input. The
function can expect that the user is passing a 2D array with attributes that are only categorical,
and the function should output a new data matrix with the converted label-encoded (integerencoded) data. [5 points]

In [ ]:
# returns int numpy array
def label_encode_2d(cat_2d):
    n, d = cat_2d.shape
    out = np.zeros((n, d), dtype=int)

    for j in range(d):
        mapping = {}      # category -> int
        next_id = 0

        for i in range(n):
            key = cat_2d[i, j]
            # convert numpy scalar to python type for dict keys
            try:
                key = key.item()
            except:
                pass

            if key not in mapping:
                mapping[key] = next_id
                next_id += 1

            out[i, j] = mapping[key]

    return out

In [ ]:
import numpy as np

# -----------------------------
# Debug comparison helpers
# -----------------------------
def max_abs_diff(A, B):
    """Return (max_diff, (i,j)) for 2D arrays or (max_diff, idx) for 1D."""
    A = np.array(A, dtype=float)
    B = np.array(B, dtype=float)

    if A.shape != B.shape:
        raise ValueError(f"Shape mismatch: {A.shape} vs {B.shape}")

    if A.ndim == 1:
        max_d = -1.0
        max_k = -1
        for k in range(A.shape[0]):
            d = abs(A[k] - B[k])
            if d > max_d:
                max_d = d
                max_k = k
        return max_d, max_k

    elif A.ndim == 2:
        n, m = A.shape
        max_d = -1.0
        max_pos = (-1, -1)
        for i in range(n):
            for j in range(m):
                d = abs(A[i, j] - B[i, j])
                if d > max_d:
                    max_d = d
                    max_pos = (i, j)
        return max_d, max_pos

    else:
        raise ValueError("Only supports 1D or 2D")


def check_close(name, got, ref, tol=1e-9):
    d, where = max_abs_diff(got, ref)
    status = "PASS" if d <= tol else "FAIL"
    print(f"[{status}] {name}: max|diff| = {d:.3e} at {where}")
    if status == "FAIL":
        print("  got:\n", np.array(got, dtype=float))
        print("  ref:\n", np.array(ref, dtype=float))
    return status == "PASS"


# -----------------------------
# Internal NumPy equivalence tests
# -----------------------------
def run_numpy_equivalence_tests(seed=0, trials=10, n_range=(5, 30), d_range=(2, 8), tol=1e-9):
    rng = np.random.default_rng(seed)
    all_ok = True

    for t in range(trials):
        n = int(rng.integers(n_range[0], n_range[1] + 1))
        d = int(rng.integers(d_range[0], d_range[1] + 1))

        # Random data; keep it float and avoid extreme magnitudes
        X = rng.normal(loc=0.0, scale=5.0, size=(n, d)).astype(float)

        print(f"\n--- Trial {t+1}/{trials}: n={n}, d={d} ---")

        # (a) mean
        mu_manual = multidimensional_mean(X)
        mu_np = np.mean(X, axis=0).reshape(1, -1)
        all_ok &= check_close("multidimensional_mean", mu_manual, mu_np, tol)

        # (d)/(h) covariance matrix
        cov_manual = covariance_matrix(X)
        cov_np = np.cov(X, rowvar=False, ddof=1)  # sample covariance (n-1)
        all_ok &= check_close("covariance_matrix", cov_manual, cov_np, tol)

        cov2_manual = compute_covariance_matrix(X)
        all_ok &= check_close("compute_covariance_matrix", cov2_manual, cov_np, tol)

        # pick two random columns for 1D tests
        i = int(rng.integers(0, d))
        j = int(rng.integers(0, d))
        while j == i:
            j = int(rng.integers(0, d))

        a = X[:, i].tolist()
        b = X[:, j].tolist()

        # (b) sample variance
        var_manual = sample_variance(a)
        var_np = np.var(X[:, i], ddof=1)
        all_ok &= check_close("sample_variance", np.array([var_manual]), np.array([var_np]), tol)

        # (c) sample covariance
        cov_ab_manual = sample_covariance(a, b)
        cov_ab_np = np.cov(X[:, i], X[:, j], ddof=1)[0, 1]
        all_ok &= check_close("sample_covariance", np.array([cov_ab_manual]), np.array([cov_ab_np]), tol)

        # (e) correlation coefficient
        corr_manual = correlation_coefficient(a, b)
        corr_np = np.corrcoef(X[:, i], X[:, j])[0, 1]
        all_ok &= check_close("correlation_coefficient", np.array([corr_manual]), np.array([corr_np]), 1e-8)

        # (f) z-score normalization (sample std)
        Z_manual = z_score_normalization(X)

        # numpy ref using sample std (ddof=1)
        mu = np.mean(X, axis=0)
        std = np.std(X, axis=0, ddof=1)
        # match your manual behavior: if std==0 => output 0
        Z_np = np.zeros_like(X, dtype=float)
        for col in range(d):
            if std[col] == 0:
                Z_np[:, col] = 0.0
            else:
                Z_np[:, col] = (X[:, col] - mu[col]) / std[col]

        all_ok &= check_close("z_score_normalization", Z_manual, Z_np, 1e-8)

        # (g) range normalization
        R_manual = range_normalization(X)

        mn = np.min(X, axis=0)
        mx = np.max(X, axis=0)
        R_np = np.zeros_like(X, dtype=float)
        for col in range(d):
            denom = mx[col] - mn[col]
            if denom == 0:
                R_np[:, col] = 0.0
            else:
                R_np[:, col] = (X[:, col] - mn[col]) / denom

        all_ok &= check_close("range_normalization", R_manual, R_np, tol)

    print("\n==============================")
    print("OVERALL:", "PASS ✅" if all_ok else "FAIL ❌ (see failures above)")
    print("==============================")
    return all_ok


# -----------------------------
# Label encoding cross-check (optional)
# -----------------------------
def run_label_encode_equivalence_test():
    """
    Note: NumPy doesn't have a direct label-encode function.
    We'll validate basic properties and a reference mapping we build the same way.
    """
    C = np.array([
        ["red",   "A"],
        ["white", "B"],
        ["red",   "B"],
        ["red",   "A"],
        ["white", "B"],
        ["blue",  "A"],
    ], dtype=object)

    E_manual = label_encode_2d(C)

    # Build a reference mapping using the SAME rule: first appearance order (per column)
    n, d = C.shape
    E_ref = np.zeros((n, d), dtype=int)
    for j in range(d):
        mapping = {}
        next_id = 0
        for i in range(n):
            key = C[i, j]
            if key not in mapping:
                mapping[key] = next_id
                next_id += 1
            E_ref[i, j] = mapping[key]

    ok = np.array_equal(E_manual, E_ref)
    print("[PASS] label_encode_2d matches reference mapping ✅" if ok else "[FAIL] label_encode_2d ❌")
    if not ok:
        print("manual:\n", E_manual)
        print("ref:\n", E_ref)
    return ok


# -----------------------------
# Run everything (internal only)
# -----------------------------
run_numpy_equivalence_tests(seed=42, trials=10, n_range=(5, 40), d_range=(2, 10), tol=1e-9)
run_label_encode_equivalence_test()

# Problem 3

**(A)** Calculate multi-dimensional mean and covariance matrix of the numerical portion of
the data.

**(B)** Convert all the categorical data attributes to numerical attributes using label encoding or
one-hot encoding.

**(C)** If your data has missing values, fill these values with attribute mean.

**(D)** Recalculate the multi-dimensional mean of the data matrix after the transformation (categorical
attributes are now numerical).

**(E)** What is the new covariance matrix of the data (categorical attributes are now numerical).

**(F)** Choose 4 pairs of numerical attributes that you think might be related. Create scatter plots for
these 4 pairs of attributes, along with the description and analysis of why these pairs of
attributes are related and how the scatter plot does or does not support your intuition.

**(G)** Which range-normalized numerical attributes have the largest estimated covariance? What is
the covariance? Create a scatter plot of these range-normalized attributes.

**(H)** Which z-score normalized numerical attributes have the largest estimated correlation
coefficient between them? What is that value? Create a scatter plot for these two z-score
normalized attributes.

**(I)** Which z-score normalized numerical attributes have the smallest estimated correlatoin coefficient between them? What is that value? Create a scatter plot for these two z-score normalized attributes.

**(J)** How many pairs of numerical features have a correlation greater than or equal to 0.5?

**(K)** How many pairs of numerical features have negative estimated covariance? 

**(L)** Calculate the total variance of the data

**(M)**  What is the ratio of the total variance of the data restricted to the five features with the highest
estimated variance to the total estimated variance of the data?